# MLB 데이터 분석 노트북
## 파크팩터 · 불펜 ERA · 모델 피처 중요도 · 예측 정확도

### 환경 설정
```bash
cd baseball-predict
python -m jupyter notebook data/notebooks/mlb_eda.ipynb
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../backend'))
from dotenv import load_dotenv
load_dotenv('../../.env')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 5)

print('환경 설정 완료')

## 1. DB 연결 및 데이터 로드

In [ ]:
import asyncio
from sqlalchemy import select, text
from app.core.database import AsyncSessionLocal

async def load_df(query: str) -> pd.DataFrame:
    async with AsyncSessionLocal() as db:
        result = await db.execute(text(query))
        rows = result.fetchall()
        cols = result.keys()
        return pd.DataFrame(rows, columns=list(cols))

# MLB 투수 스탯
df_pitch = await load_df("""
    SELECT * FROM mlb_pitcher_stats
    ORDER BY season DESC, ip DESC
""")

# MLB 팀 불펜
df_bullpen = await load_df("""
    SELECT * FROM mlb_team_bullpen_stats ORDER BY season DESC, bullpen_era
""")

# MLB 팀 타선
df_batting = await load_df("""
    SELECT * FROM mlb_team_batting_stats ORDER BY season DESC, ops DESC
""")

# 예측 정확도
df_pred = await load_df("""
    SELECT p.*, g.league, g.game_date, g.home_score, g.away_score,
           ht.short_name AS home_team, at.short_name AS away_team
    FROM predictions p
    JOIN games g ON p.game_id = g.id
    JOIN teams ht ON g.home_team_id = ht.id
    JOIN teams at ON g.away_team_id = at.id
    WHERE g.league = 'MLB' AND p.was_correct IS NOT NULL
    ORDER BY g.game_date DESC
""")

print(f'투수: {len(df_pitch)}명, 불펜: {len(df_bullpen)}팀, 타선: {len(df_batting)}팀, 예측: {len(df_pred)}건')

## 2. MLB 구장별 파크팩터 시각화

In [ ]:
df_teams = await load_df("""
    SELECT short_name, stadium_name, park_factor
    FROM teams WHERE league = 'MLB'
    ORDER BY park_factor DESC
""")

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#e74c3c' if pf > 1.0 else '#3498db' for pf in df_teams['park_factor']]
bars = ax.barh(df_teams['short_name'], df_teams['park_factor'], color=colors, edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='중립(1.0)')
ax.set_xlabel('파크팩터 (>1.0 = 타자 유리)', fontsize=12)
ax.set_title('MLB 30개 구단 파크팩터', fontsize=14, fontweight='bold')
for bar, val in zip(bars, df_teams['park_factor']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 3. 팀 불펜 ERA 분포 (2024 vs 2025)

In [ ]:
if df_bullpen.empty:
    print('불펜 데이터 없음 — python collect_mlb_stats.py --all 실행 후 재시도')
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, season in zip(axes, [2024, 2025]):
        data = df_bullpen[df_bullpen['season'] == season].sort_values('bullpen_era')
        if data.empty:
            ax.set_title(f'{season} 데이터 없음')
            continue
        colors = ['#e74c3c' if e > data['bullpen_era'].median() else '#2ecc71' for e in data['bullpen_era']]
        ax.bar(data['team_short'], data['bullpen_era'], color=colors)
        ax.axhline(data['bullpen_era'].mean(), color='black', linestyle='--', label=f'평균 {data["bullpen_era"].mean():.2f}')
        ax.set_title(f'{season} 팀 불펜 ERA', fontsize=13, fontweight='bold')
        ax.set_xticklabels(data['team_short'], rotation=90, fontsize=8)
        ax.legend()
    plt.tight_layout()
    plt.show()

## 4. 선발투수 ERA vs FIP 산포도

In [ ]:
if df_pitch.empty:
    print('투수 데이터 없음')
else:
    for season in df_pitch['season'].unique():
        sp = df_pitch[(df_pitch['season'] == season) & (df_pitch['gs'] >= 5) & (df_pitch['fip'].notna())]
        if sp.empty:
            continue
        fig, ax = plt.subplots(figsize=(10, 7))
        sc = ax.scatter(sp['era'], sp['fip'], c=sp['ip'], cmap='RdYlGn_r',
                        s=60, alpha=0.7, edgecolors='white')
        # ERA = FIP 기준선
        lims = [min(sp['era'].min(), sp['fip'].min()) - 0.2,
                max(sp['era'].max(), sp['fip'].max()) + 0.2]
        ax.plot(lims, lims, 'k--', alpha=0.4, label='ERA = FIP')
        # 아웃라이어 선수명 표시
        diff = (sp['era'] - sp['fip']).abs()
        top5 = sp.nlargest(5, 'ip') if len(sp) > 5 else sp
        for _, row in top5.iterrows():
            ax.annotate(row['name'].split()[-1], (row['era'], row['fip']),
                        fontsize=7, xytext=(4, 4), textcoords='offset points')
        plt.colorbar(sc, ax=ax, label='이닝(IP)')
        ax.set_xlabel('ERA', fontsize=12)
        ax.set_ylabel('FIP', fontsize=12)
        ax.set_title(f'{season} 선발투수 ERA vs FIP (버블=이닝)', fontsize=13, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.show()

## 5. 투구방향별 ERA 분포 (좌완 vs 우완)

In [ ]:
if not df_pitch.empty:
    sp = df_pitch[(df_pitch['gs'] >= 5) & (df_pitch['handedness'].notna())]
    fig, ax = plt.subplots()
    for hand, label, color in [('L', '좌완 (LHP)', '#3498db'), ('R', '우완 (RHP)', '#e74c3c')]:
        data = sp[sp['handedness'] == hand]['era'].dropna()
        ax.hist(data, bins=20, alpha=0.6, label=f'{label} (n={len(data)})', color=color)
    ax.set_xlabel('ERA', fontsize=12)
    ax.set_ylabel('선수 수', fontsize=12)
    ax.set_title('선발투수 ERA 분포 — 좌완 vs 우완', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. 예측 정확도 분석 (신뢰도 등급별)

In [ ]:
if df_pred.empty:
    print('예측 데이터 없음')
else:
    df_pred['was_correct'] = df_pred['was_correct'].astype(bool)
    # 신뢰도별 정확도
    by_tier = df_pred.groupby('confidence_tier')['was_correct'].agg(['mean', 'count'])
    by_tier.columns = ['정확도', '예측수']
    by_tier = by_tier.reindex(['high', 'medium', 'low'])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 정확도
    ax = axes[0]
    colors = {'high': '#2ecc71', 'medium': '#f39c12', 'low': '#e74c3c'}
    by_tier['정확도'].plot(kind='bar', ax=ax,
                         color=[colors.get(t, 'gray') for t in by_tier.index])
    ax.axhline(0.5, linestyle='--', color='black', label='50% (무작위)')
    ax.set_title('신뢰도 등급별 예측 정확도', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_xticklabels(['높음', '중간', '낮음'], rotation=0)
    for i, (acc, cnt) in enumerate(zip(by_tier['정확도'], by_tier['예측수'])):
        ax.text(i, acc + 0.01, f'{acc:.1%}\n(n={int(cnt)})', ha='center', fontsize=10)
    ax.legend()

    # 홈팀 승리 확률 vs 결과 calibration
    ax = axes[1]
    bins = np.arange(0.3, 0.85, 0.05)
    df_pred['prob_bin'] = pd.cut(df_pred['home_win_prob'], bins=bins)
    cal = df_pred.groupby('prob_bin', observed=True)['was_correct'].agg(['mean', 'count'])
    cal_x = [interval.mid for interval in cal.index]
    ax.scatter(cal_x, cal['mean'], s=cal['count'] * 2, alpha=0.7, color='#3498db')
    ax.plot([0.3, 0.85], [0.3, 0.85], 'k--', alpha=0.4, label='완벽한 캘리브레이션')
    ax.set_xlabel('예측 홈팀 승리 확률', fontsize=12)
    ax.set_ylabel('실제 정확도', fontsize=12)
    ax.set_title('예측 확률 캘리브레이션', fontsize=13, fontweight='bold')
    ax.legend()

    plt.tight_layout()
    plt.show()
    print(f'전체 정확도: {df_pred["was_correct"].mean():.1%} (n={len(df_pred)})')

## 7. 피처 중요도 (XGBoost SHAP)

In [ ]:
from pathlib import Path
import xgboost as xgb
import shap
from app.features.builder import get_feature_columns
from app.config import settings

model_dir = Path(settings.model_path)
candidates = sorted(model_dir.glob('xgb-mlb-v*.ubj'), reverse=True)

if not candidates:
    print('MLB XGBoost 모델 없음 — POST /api/v1/admin/retrain 으로 재학습 후 재실행')
else:
    model = xgb.XGBClassifier()
    model.load_model(str(candidates[0]))
    print(f'모델 로드: {candidates[0].name}')

    feature_cols = get_feature_columns('MLB')
    
    # 내장 feature importance
    importances = model.feature_importances_
    fi_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors_fi = ['#e74c3c' if i < 5 else '#3498db' for i in range(len(fi_df))]
    ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color=colors_fi[::-1])
    ax.set_xlabel('피처 중요도 (XGBoost gain)', fontsize=12)
    ax.set_title('MLB 예측 모델 — 상위 20개 피처 중요도', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('\n상위 10개 피처:')
    print(fi_df.head(10).to_string(index=False))

## 8. 불펜 ERA vs 팀 승률 상관관계

In [ ]:
if df_bullpen.empty or df_pred.empty:
    print('데이터 부족')
else:
    # 팀별 홈 승률 계산
    home_wins = df_pred.groupby('home_team').apply(
        lambda x: pd.Series({'win_rate': x['was_correct'].mean(), 'games': len(x)})
    ).reset_index()
    
    # 불펜 ERA와 merge
    bp_latest = df_bullpen.sort_values('season', ascending=False).drop_duplicates('team_short')
    merged = home_wins.merge(bp_latest[['team_short', 'bullpen_era']], 
                              left_on='home_team', right_on='team_short', how='inner')
    
    if len(merged) > 5:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(merged['bullpen_era'], merged['win_rate'],
                   s=merged['games'] * 3, alpha=0.7, color='#3498db', edgecolors='white')
        for _, row in merged.iterrows():
            ax.annotate(row['home_team'], (row['bullpen_era'], row['win_rate']),
                        fontsize=8, xytext=(4, 4), textcoords='offset points')
        z = np.polyfit(merged['bullpen_era'], merged['win_rate'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(merged['bullpen_era'].min(), merged['bullpen_era'].max(), 100)
        ax.plot(x_line, p(x_line), 'r--', alpha=0.5, label=f'추세선')
        corr = merged['bullpen_era'].corr(merged['win_rate'])
        ax.set_xlabel('팀 불펜 ERA', fontsize=12)
        ax.set_ylabel('홈경기 예측 정확도', fontsize=12)
        ax.set_title(f'불펜 ERA vs 예측 정확도 (상관계수: {corr:.3f})', fontsize=13, fontweight='bold')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print(f'샘플 부족 ({len(merged)}개)')

## 9. 구장별 예측 정확도 (파크팩터 영향)

In [ ]:
if not df_pred.empty:
    teams_pf = await load_df("SELECT short_name, park_factor FROM teams WHERE league='MLB'")
    
    venue_acc = df_pred.groupby('home_team')['was_correct'].agg(['mean', 'count']).reset_index()
    venue_acc.columns = ['home_team', 'accuracy', 'count']
    venue_acc = venue_acc[venue_acc['count'] >= 5]  # 최소 5경기
    venue_acc = venue_acc.merge(teams_pf, left_on='home_team', right_on='short_name', how='inner')

    if len(venue_acc) > 5:
        fig, ax = plt.subplots(figsize=(10, 6))
        sc = ax.scatter(venue_acc['park_factor'], venue_acc['accuracy'],
                        s=venue_acc['count'] * 5, alpha=0.7, c=venue_acc['accuracy'],
                        cmap='RdYlGn', vmin=0.4, vmax=0.7, edgecolors='white')
        for _, row in venue_acc.iterrows():
            ax.annotate(row['home_team'], (row['park_factor'], row['accuracy']),
                        fontsize=8, xytext=(4, 4), textcoords='offset points')
        ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5)
        plt.colorbar(sc, ax=ax, label='정확도')
        ax.set_xlabel('파크팩터', fontsize=12)
        ax.set_ylabel('예측 정확도', fontsize=12)
        ax.set_title('구장 파크팩터 vs 예측 정확도', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()